In [1]:
import os
import gc
import zarr
import yaml
import json
import numba
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from anngeno import AnnGeno
import multiprocessing

import matplotlib.pyplot as plt
from plotnine import *

num_cores = multiprocessing.cpu_count()
print(num_cores)

16


In [2]:
def process_phenotypes_prs_long(
    pheno_lazy: str,
    prs_lazy: str,
    cov_lazy: str,
    unique_phenotypes: pl.DataFrame,
    cov_list: list,
    quantitative: bool = True
) -> pl.DataFrame:
    """
    Process phenotypes and PRS, compute residuals for each phenotype, and return a long-format Polars DataFrame.
    """
    print("Process phenotypes and PRS, compute residuals for each phenotype, and return a long-format Polars DataFrame.")

    # --- Step 3: Merge all into one lazy DataFrame ---
    all_lazy = pheno_lazy.join(prs_lazy, on='individual', how='inner').join(cov_lazy, on='individual', how='inner')
    
    # Collect once for regression computations (still needed for statsmodels)
    all_pd = all_lazy.collect().to_pandas()

    # --- Step 4: Compute residuals ---
    all_residuals_dfs = []

    for phenotype in tqdm(unique_phenotypes):
        pheno_cols = [phenotype, f"{phenotype}_prs"] + cov_list
        temp_df = all_pd[['individual'] + pheno_cols].dropna()
        if len(temp_df) == 0:
            print(f"No data for phenotype: {phenotype}")
            continue

        y = temp_df[phenotype]
        X = temp_df.drop(columns=[phenotype, 'individual'])
        X = sm.add_constant(X)

        if quantitative:
            model = sm.OLS(y, X).fit()
            residuals = pd.Series(model.resid, index=temp_df.index, name=f"{phenotype}_residual")
        else:
            model = sm.GLM(y, X, family=sm.families.Binomial()).fit()
            residuals = pd.Series(model.resid_deviance, index=temp_df.index, name=f"{phenotype}_residual")

        pheno_residuals = pd.concat([temp_df[['individual']], residuals], axis=1)
        all_residuals_dfs.append(pheno_residuals)


    if not all_residuals_dfs:
        raise ValueError("No residuals could be computed")

    # --- Step 5: Convert to long-format lazy DataFrame ---
    long_lazy_dfs = []
    for residual_df in all_residuals_dfs:
        p_wide_lazy = pl.LazyFrame(residual_df).with_columns(
            pl.col('individual').cast(pl.String)
        )
        pheno_cols = [c for c in residual_df.columns if c.endswith('_residual')]
        pdf_lazy = (
            p_wide_lazy.unpivot(
                index=['individual'],
                on=pheno_cols,
                variable_name='phenotype',
                value_name='pheno_value',
            )
            .with_columns(
                pl.col('phenotype').str.replace('_residual', '').alias('phenotype')
            )
        )
        long_lazy_dfs.append(pdf_lazy)

    combined_pdf_lazy = pl.concat(long_lazy_dfs) if len(long_lazy_dfs) > 1 else long_lazy_dfs[0]

    return combined_pdf_lazy

In [3]:
@numba.njit(parallel=True, fastmath=True)
def _fast_clip_and_sum_allels(arr):
    # Get the shape of the input array
    # Using specific dimensions for clarity with this problem
    n_samples, n_variants, _ = arr.shape
    
    # The sum of two positive int8s can be up to 254. 
    # An int16 is a safe and fast output type.
    output = np.empty((n_samples, n_variants), dtype=np.int8)
    
    # Numba's prange enables automatic parallelization across all your CPU cores
    for i in numba.prange(n_samples):
        for j in range(n_variants):
            # Read two values, perform logic, write one value.
            # This is the "fused" operation.
            val1 = arr[i, j, 0]
            val2 = arr[i, j, 1]
            
            s = 0
            # Since input is int8, this check is faster than max(0, val)
            if val1 > 0:
                s += val1
            if val2 > 0:
                s += val2
            
            output[i, j] = s
            
    return output

In [4]:
def process_genotype_chunk(
    geno: np.array, 
    var_ids: np.array, 
    sample_list: np.array,
    melted_pheno_df: pl.LazyFrame,
    homozygous: bool = False,
    debug: bool = False,
) -> pl.LazyFrame:
    """
    Extract genotypes for a specific gene and return as lazy DataFrame
    """
    geno_clipped = _fast_clip_and_sum_allels(geno)
    # Find heterozygous genotypes (genotype == 1)
    rows, cols = np.where(geno_clipped == 1)
    geno_melt = pl.DataFrame({
        'id': var_ids[rows],
        'individual': sample_list[cols],
        'genotype': 1
    })
    
    # Find homozygous genotypes (genotype == 2)
    if homozygous:
        rows, cols = np.where(geno_clipped == 2)
        hom = pl.DataFrame({
            'id': var_ids[rows],
            'individual': sample_list[cols],
            'genotype': 2
        })
        geno_melt = pl.concat([geno_melt, hom])

    var_pheno_df = geno_melt.lazy().join(melted_pheno_df, on='individual', how='left')
    if debug:
        # Return intermediate dataframe if debugging
        return var_pheno_df
    
    var_pheno_df = var_pheno_df.group_by(
           ['id', 'phenotype']
           ).agg([
                pl.len().alias('n_individuals'),
                pl.col('pheno_value').mean().cast(pl.Float32).alias('mean_pheno_value'),
                pl.col('pheno_value').std().cast(pl.Float32).alias('std_pheno_value'),
            ]).drop_nulls(subset=['mean_pheno_value'])
    return var_pheno_df


In [5]:
pheno_path = '/home/dnanexus/data_dir/phenotypes/phenotypes190_missing20_unique2_int.parquet'
prs_path = '/home/dnanexus/data_dir/phenotypes/PRS190_EUR_missing20_unique2_int.parquet'
cov_path = '/home/dnanexus/data_dir/phenotypes/250709_quant_phenotypes_covariates_genetic_pcs_prs_corrected.parquet'

anngeno_path = '/home/dnanexus/data_dir/genebass_1e6_coding_variants.ag'
eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'

In [6]:
sample_ids = zarr.open(f'{anngeno_path}/zarr_store/samples', mode='r')[:]
sample_ids

/home/dnanexus/deeprvat2-env/lib/python3.11/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.


array(['W000001', 'W000002', 'W000003', ..., '3548117', '1732136',
       '5618419'], shape=(490541,), dtype=StringDType())

In [7]:
var_ids = pl.read_parquet(f'{anngeno_path}/variant_metadata.parquet', columns=['id'])['id'].to_numpy()
var_ids

array(['chr1:1373801:T:C', 'chr1:1373805:T:C', 'chr1:1373806:T:C', ...,
       'chr22:50556402:C:A', 'chr22:50556404:A:G', 'chr22:50556407:T:G'],
      shape=(1826101,), dtype=object)

In [8]:
geno = zarr.open(f'{anngeno_path}/zarr_store/genotypes', mode='r')
geno[:5].shape

(5, 490541, 2)

In [9]:
eur_samples = pl.read_csv(eur_samples_path).rename({'eid': 'individual'}).with_columns(
    pl.col("individual").cast(pl.Utf8)
)['individual'].to_list()

pheno_lazy = pl.read_parquet(pheno_path).drop(['FID']).rename({'IID': 'individual'}).with_columns(
    pl.col("individual").cast(pl.Utf8)
).filter(
    pl.col("individual").is_in(eur_samples)
).fill_nan(None).lazy()
# .unpivot(
#     index=['individual'],
#     variable_name='phenotype',
#     value_name='pheno_value',
# ).drop_nulls().lazy()

pheno_lazy.head().collect()

individual,red_blood_cell_erythrocyte_distribution_width_int,mean_platelet_thrombocyte_volume_int,mean_time_to_correctly_identify_matches_int,mothers_age_at_death_int,systolic_blood_pressure_automated_reading_int,microalbumin_in_urine_int,cholesterol_int,haemoglobin_concentration_int,aspartate_aminotransferase_int,alanine_aminotransferase_int,forced_expiratory_volume_in_1second_fev1_predicted_int,age_started_hormonereplacement_therapy_hrt_int,hand_grip_strength_left_int,vitamin_d_int,mean_corpuscular_haemoglobin_concentration_int,fathers_age_at_death_int,arm_predicted_mass_left_int,mean_corpuscular_haemoglobin_int,position_of_pulse_wave_notch_int,augmentation_index_for_pwa_int,leg_fat_percentage_right_int,leg_predicted_mass_right_int,forced_expiratory_volume_in_1second_fev1_predicted_percentage_int,basal_metabolic_rate_int,alkaline_phosphatase_int,weight_int,lymphocyte_percentage_int,basophill_percentage_int,leg_predicted_mass_left_int,total_protein_int,pulse_rate_int,age_started_wearing_glasses_or_contact_lenses_int,lipoprotein_a_int,peak_expiratory_flow_pef_int,creatinine_int,direct_bilirubin_int,…,jurgens_cardiac_surgery,jurgens_av_or_bundle_branch_block,jurgens_cervical_cancer,jurgens_breast_cancer,jurgens_hypercholesterolemia,jurgens_colorectal_cancer,jurgens_migraine,jurgens_sleep_apnea,jurgens_parkinsons_disease,jurgens_dermatitis,jurgens_skin_cancer,jurgens_stroke,jurgens_venous_thromboembolism,jurgens_myocardial_infarction,jurgens_cataract,jurgens_depression,jurgens_pancreatitis,jurgens_lung_cancer,jurgens_hyperthyroidism,jurgens_bipolar_disorder,jurgens_osteoarthritis,jurgens_hypertrophic_cardiomyopathy,jurgens_inflammatory_bowel_disease,jurgens_ventricular_arrhythmia,jurgens_asthma,jurgens_ischemic_stroke,jurgens_cardiac_arrest,jurgens_anxiety,jurgens_hypertension,jurgens_coronary_artery_disease,jurgens_supraventricular_tachycardia,jurgens_pneumonia,jurgens_rheumatoid_arthritis,jurgens_heart_failure,jurgens_prostate_cancer,jurgens_epilepsy,jurgens_bladder_cancer
str,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32
"""1000018""",-0.523548,-1.114154,-1.499334,null,-0.7141,null,-0.857422,0.647891,0.87209,1.605751,1.229859,null,2.887276,0.424981,-0.525447,null,1.326517,0.535457,null,null,-0.318126,0.861412,0.282365,1.065923,0.325578,0.834738,1.231852,0.238699,0.972997,-0.882812,null,null,-1.462396,1.204289,0.282418,0.864511,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""1000020""",1.297892,-0.20989,-1.363095,-1.357539,-0.00655,null,1.239516,0.56987,0.378716,-0.567495,1.215144,null,0.691104,-0.475926,-0.291717,null,0.604962,-0.202377,null,null,-1.70112,0.570342,0.77198,0.471579,-0.966527,-0.182016,-1.134633,1.100623,0.490911,-0.970244,null,0.351249,-0.149689,1.124732,-0.081447,null,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""1000037""",0.890615,0.036805,0.3741,null,-0.542032,0.189598,-2.323255,1.151392,-1.060129,-1.217438,null,null,1.165187,1.750258,0.19216,0.362836,0.717834,0.84297,null,null,-1.964797,0.570342,null,0.618321,-0.38591,-0.175717,-0.962164,-0.154609,0.490911,0.689715,null,null,-0.495531,0.869697,0.359893,-0.585654,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""1000043""",-0.685765,-1.047474,0.566844,1.661004,0.901054,null,0.729993,-0.926205,-0.69629,-1.64525,null,-0.505974,-1.547315,0.481257,-1.253048,1.211783,-1.458727,-1.006978,null,null,0.197221,-2.126477,null,-1.981572,1.255572,-1.980985,0.274264,-0.154609,-2.198781,null,null,0.16251,0.410834,-2.01

In [10]:
# unique_phenotypes = pheno_df.select('phenotype').unique().collect()['phenotype'].to_list()
unique_phenotypes = [pheno for pheno in pheno_lazy.columns if pheno != 'individual']
quant_phenotypes = [pheno for pheno in unique_phenotypes if "jurgens" not in pheno]

prs_cols = [f"{pheno}_prs" for pheno in quant_phenotypes]
prs_lazy = (
    pl.scan_parquet(prs_path)
    .fill_nan(None)
    .rename({'IID': 'individual'})
    .select(['individual'] + prs_cols)
    .with_columns(pl.col("individual").cast(pl.Utf8))
    .drop_nulls()
)

config_path = f'/home/dnanexus/ukbgym/config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

cov_list = config.get("covariates", [])
cov_lazy = (
    pl.scan_parquet(cov_path)
    .rename({'sample': 'individual'})
    .select(['individual'] + cov_list)
)

/tmp/ipykernel_597721/4254716283.py:2: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.


In [11]:
corr_phenos_lazy = process_phenotypes_prs_long(
    pheno_lazy=pheno_lazy,
    prs_lazy=prs_lazy,
    cov_lazy=cov_lazy,
    unique_phenotypes=quant_phenotypes,
    cov_list=cov_list,
    quantitative=True
)

corr_phenos_lazy.head().collect()

Process phenotypes and PRS, compute residuals for each phenotype, and return a long-format Polars DataFrame.


100%|██████████| 127/127 [00:48<00:00,  2.61it/s]


individual,phenotype,pheno_value
str,str,f64
"""1000020""","""red_blood_cell_erythrocyte_dis…",0.595553
"""1000107""","""red_blood_cell_erythrocyte_dis…",0.002036
"""1000161""","""red_blood_cell_erythrocyte_dis…",-0.68081
"""1000172""","""red_blood_cell_erythrocyte_dis…",-0.254212
"""1000221""","""red_blood_cell_erythrocyte_dis…",-0.237714


In [12]:
output_dir = "/home/dnanexus/data_dir/var_pheno_EUR_chunks/"
chunk_size = 10_000

for chunk_num in tqdm(range(var_ids.shape[0]//chunk_size + 1)):
    process_genotype_chunk(
        geno=geno[chunk_num*chunk_size:(chunk_num+1)*chunk_size],
        var_ids=var_ids[chunk_num*chunk_size:(chunk_num+1)*chunk_size],
        sample_list=sample_ids,
        melted_pheno_df=corr_phenos_lazy,
        homozygous=False,
    ).sink_parquet(f"{output_dir}/variant_pheno_chunk{chunk_num}.parquet")

100%|██████████| 183/183 [1:16:05<00:00, 24.95s/it]


In [13]:
files = [f"{output_dir}/variant_pheno_chunk{i}.parquet" for i in range(var_ids.shape[0]//chunk_size + 1)]

# Create lazy frames (scans don't load data yet)
lazy_frames = [pl.scan_parquet(f) for f in files]

# Concatenate lazily
combined = pl.concat(lazy_frames)

# Now you can collect, or better: sink to Parquet directly
combined.sink_parquet(f"{output_dir}/variant_pheno_EUR.parquet")

In [21]:
vp = pl.scan_parquet(f"{output_dir}/variant_pheno_EUR.parquet")
vp.select('id').collect().n_unique()

1210768

In [22]:
vp.select('phenotype').collect()['phenotype'].value_counts(sort=True)

phenotype,count
str,u32
"""townsend_deprivation_index_at_…",1210064
"""waist_circumference_int""",1209839
"""hip_circumference_int""",1209783
"""standing_height_int""",1209573
"""seated_height_int""",1209518
…,…
"""oestradiol_int""",493980
"""age_at_hysterectomy_int""",404423
"""augmentation_index_for_pwa_int""",356696


## Debug code

In [52]:
n_vars = 2
vp = process_genotype_chunk(
    geno=geno[:n_vars],
    var_ids=var_ids[:n_vars],
    sample_list=sample_ids,
    melted_pheno_df=pheno_lazy.filter(pl.col('phenotype').is_in(quant_phenotypes)),
    homozygous=False,
    debug=True,
).collect()

vp

id,individual,genotype,phenotype,pheno_value
str,str,i32,str,f32
"""chr1:1373801:T:C""","""3594985""",1,null,null
"""chr1:1373805:T:C""","""4610444""",1,null,null
"""chr1:1373805:T:C""","""3240568""",1,"""red_blood_cell_erythrocyte_dis…",1.732442
"""chr1:1373805:T:C""","""3240568""",1,"""mean_platelet_thrombocyte_volu…",-0.010088
"""chr1:1373805:T:C""","""3240568""",1,"""mean_time_to_correctly_identif…",-1.196428
…,…,…,…,…
"""chr1:1373805:T:C""","""2864227""",1,"""leg_fat_percentage_left_int""",-1.906088
"""chr1:1373805:T:C""","""2864227""",1,"""glycated_haemoglobin_hba1c_int""",1.222078
"""chr1:1373805:T:C""","""2864227""",1,"""total_bilirubin_int""",-0.063124


In [18]:
n_vars = 100
vp = process_genotype_chunk(
    geno=geno[:n_vars],
    var_ids=var_ids[:n_vars],
    sample_list=sample_ids,
    melted_pheno_df=corr_phenos_lazy,
    homozygous=False,
    debug=True,
).collect()

vp

id,individual,genotype,phenotype,pheno_value
str,str,i32,str,f64
"""chr1:1373801:T:C""","""3594985""",1,null,null
"""chr1:1373805:T:C""","""4610444""",1,null,null
"""chr1:1373805:T:C""","""3240568""",1,"""red_blood_cell_erythrocyte_dis…",1.268318
"""chr1:1373805:T:C""","""3240568""",1,"""mean_platelet_thrombocyte_volu…",0.3101
"""chr1:1373805:T:C""","""3240568""",1,"""mean_time_to_correctly_identif…",-1.594848
…,…,…,…,…
"""chr1:1374071:A:G""","""3592825""",1,"""glycated_haemoglobin_hba1c_int""",-0.610136
"""chr1:1374071:A:G""","""3592825""",1,"""total_bilirubin_int""",-0.685119
"""chr1:1374071:A:G""","""3592825""",1,"""sitting_height_int""",-0.434304


In [40]:
vp.group_by(
    ['id', 'phenotype']
).agg([
    pl.len().alias('n_individuals'),
    pl.col('pheno_value').mean().alias('mean_pheno_value'),
    pl.col('pheno_value').std().alias('std_pheno_value'),
]).drop_nulls(subset=['mean_pheno_value'])

id,phenotype,n_individuals,mean_pheno_value,std_pheno_value
str,str,u32,f32,f32
"""chr1:1373805:T:C""","""systolic_blood_pressure_automa…",1558,0.045339,0.985217
"""chr1:1373805:T:C""","""neutrophill_count_int""",1622,0.054701,1.00144
"""chr1:1373805:T:C""","""albumin_int""",1462,0.071365,0.971099
"""chr1:1373805:T:C""","""jurgens_dermatitis""",1591,0.050283,0.218597
"""chr1:1373805:T:C""","""jurgens_chronic_obstructive_pu…",1591,0.084224,0.27781
…,…,…,…,…
"""chr1:1373805:T:C""","""igf1_int""",1584,0.007895,0.957842
"""chr1:1373805:T:C""","""heel_bone_mineral_density_bmd_…",553,-0.048408,0.988121
"""chr1:1373805:T:C""","""birth_weight_int""",961,0.072232,1.027018


In [41]:
%%time

n_vars = 1_000
vp = process_genotype_chunk(
    geno=geno[:n_vars],
    var_ids=var_ids[:n_vars],
    sample_list=sample_ids,
    melted_pheno_df=pheno,
    homozygous=False,
).collect()


CPU times: user 20.4 s, sys: 1.61 s, total: 22 s
Wall time: 3.55 s


In [ ]:
%%time

n_vars = 10_000
vp = process_genotype_chunk(
    geno=geno[:n_vars],
    var_ids=var_ids[:n_vars],
    sample_list=sample_ids,
    melted_pheno_df=pheno,
    homozygous=False,
).collect()

CPU times: user 3min 22s, sys: 19.8 s, total: 3min 42s
Wall time: 41.8 s


: 

In [ ]:
%%time

n_vars = 20_000
vp = process_genotype_chunk(
    geno=geno[:n_vars],
    var_ids=var_ids[:n_vars],
    sample_list=sample_ids,
    melted_pheno_df=pheno,
    homozygous=False,
).collect()

In [ ]:
(1826101/10_000)*17.5